In [1]:
import os
import zipfile

from pathlib import Path

import requests

# Setup path to data folder
data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"

# If the image folder doesn't exist, download it and prepare it...
if image_path.is_dir():
    print(f"{image_path} directory exists.")
else:
    print(f"Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)

# Download pizza, steak, sushi data
with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
    request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
    print("Downloading pizza, steak, sushi data...")
    f.write(request.content)

# Unzip pizza, steak, sushi data
with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", "r") as zip_ref:
    print("Unzipping pizza, steak, sushi data...")
    zip_ref.extractall(image_path)

# Remove zip file
os.remove(data_path / "pizza_steak_sushi.zip")

Did not find data/pizza_steak_sushi directory, creating one...
Unzipping pizza, steak, sushi data...


In [2]:
# Setup train and testing paths
train_dir = image_path / "train"
test_dir = image_path / "test"

train_dir, test_dir

(PosixPath('data/pizza_steak_sushi/train'),
 PosixPath('data/pizza_steak_sushi/test'))

In [3]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets,transforms

In [4]:
# Write a transform for image
data_transform = transforms.Compose([
    #Resize our image to 64x64
    transforms.Resize(size=(64,64)),
    #Flip the image randomly on the horizontal
    transforms.RandomHorizontalFlip(p=0.5),
    #Turn the image into the torch.tensor
    transforms.ToTensor()
])

In [5]:
# Use `image folder` to create the datasets
# ImageFolder automatically assigns labels based on subfolder names

from torchvision import datasets
train_data = datasets.ImageFolder(root=train_dir,
                                  transform=data_transform, # Transform for the data
                                  target_transform=None) # transform for the label / target

test_data = datasets.ImageFolder(root=test_dir,
                                 transform=data_transform,
                                 target_transform=None)

train_data,test_data

(Dataset ImageFolder
     Number of datapoints: 225
     Root location: data/pizza_steak_sushi/train
     StandardTransform
 Transform: Compose(
                Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
                RandomHorizontalFlip(p=0.5)
                ToTensor()
            ),
 Dataset ImageFolder
     Number of datapoints: 75
     Root location: data/pizza_steak_sushi/test
     StandardTransform
 Transform: Compose(
                Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
                RandomHorizontalFlip(p=0.5)
                ToTensor()
            ))

In [6]:
# Get classs names

class_names = train_data.classes#ImageFolder's function


len(class_names),class_names

(3, ['pizza', 'steak', 'sushi'])

In [7]:
# 2. Turn data into DataLoaders
import os
from torch.utils.data import DataLoader

# Setup batch size and number of workers
BATCH_SIZE = 32
NUM_WORKERS = 2

# Create the dataloader
train_dataloader = DataLoader(
                                      dataset=train_data,
                                      batch_size= BATCH_SIZE,
                                      shuffle= True,
                                      num_workers=NUM_WORKERS #有多少個子程序（subprocesses）平行載入與預處理資料。
)

test_dataloader = DataLoader(
                                      dataset=train_data,
                                      batch_size= BATCH_SIZE,
                                      shuffle= False,
                                      num_workers=NUM_WORKERS
)

train_dataloader,test_dataloader

(<torch.utils.data.dataloader.DataLoader at 0x7b75831c9a90>,
 <torch.utils.data.dataloader.DataLoader at 0x7b758338f250>)

## 2. Create Datasets and DataLoaders

Now we'll turn image dataset into PyTorch `Dataset`'s and `DataLoader`'s

In [8]:
import os
os.makedirs("going_modular")

In [9]:
%%writefile going_modular/data_setup.py
"""
 modular script
"""
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os
import torch


numworkers=os.cpu_count()


def create_dataloaders(train_dir, test_dir, transform, batch_size, num_workers=numworkers):
  """Create PyTorch DataLoaders for training and testing datasets.

    This function loads data from the specified training and testing directories,
    applies the given transforms, and constructs DataLoader instances for each set.

    Args:
        train_dir (str or Path): Path to the training data directory.
        test_dir (str or Path): Path to the testing data directory.
        transform (callable): Transformations to apply to the datasets.
        batch_size (int): Number of samples per batch.
        num_workers (int): Number of subprocesses to use for data loading.

    Returns:
        tuple[
            torch.utils.data.DataLoader,
            torch.utils.data.DataLoader,
            list[str]
        ]:
            A tuple containing the training DataLoader, testing DataLoader,
            and a list of class names.
  """
  # Create datasets
  train_data = datasets.ImageFolder(train_dir, transform=transform)
  test_data = datasets.ImageFolder(test_dir, transform=transform)

  # Get class names
  class_names = train_data.classes

  # Create dataloaders
  train_dataloader = DataLoader(
      train_data,
      batch_size=batch_size,
      shuffle=True,
      num_workers=numworkers,
      pin_memory=True,
  )
  test_dataloader = DataLoader(
      test_data,
      batch_size=batch_size,
      shuffle=False,
      num_workers=numworkers,
      pin_memory=True,
  )

  return train_dataloader, test_dataloader, class_names

Writing going_modular/data_setup.py


In [10]:
from going_modular import data_setup

help(data_setup.create_dataloaders)



Help on function create_dataloaders in module going_modular.data_setup:

create_dataloaders(train_dir, test_dir, transform, batch_size, num_workers=2)
    Create PyTorch DataLoaders for training and testing datasets.

    This function loads data from the specified training and testing directories,
    applies the given transforms, and constructs DataLoader instances for each set.

    Args:
        train_dir (str or Path): Path to the training data directory.
        test_dir (str or Path): Path to the testing data directory.
        transform (callable): Transformations to apply to the datasets.
        batch_size (int): Number of samples per batch.
        num_workers (int): Number of subprocesses to use for data loading.

    Returns:
        tuple[
            torch.utils.data.DataLoader,
            torch.utils.data.DataLoader,
            list[str]
        ]:
            A tuple containing the training DataLoader, testing DataLoader,
            and a list of class names.



In [11]:
%%writefile going_modular/model_builder.py


import torch

from torch import nn

class TinyVGG(nn.Module):
  """Creates the TinyVGG architecture.
  Replicates the TinyVGG architecture from the CNN explainer website in PyTorch.
  See the original architecture here: https://poloclub.github.io/cnn-explainer/

  Args:
    input_shape: An integer indicating number of input channels.
    hidden_units: An integer indicating number of hidden units between layers.
    output_shape: An integer indicating number of output units.
  """
  def __init__(self, input_shape: int, hidden_units: int, output_shape: int) -> None:
      super().__init__()
      self.conv_block_1 = nn.Sequential(
          nn.Conv2d(in_channels=input_shape,
                    out_channels=hidden_units,
                    kernel_size=3, # how big is the square that's going over the image?
                    stride=1, # default
                    padding=0), # options = "valid" (no padding) or "same" (output has same shape as input) or int for specific number
          nn.ReLU(),
          nn.Conv2d(in_channels=hidden_units,
                    out_channels=hidden_units,
                    kernel_size=3,
                    stride=1,
                    padding=0),
          nn.ReLU(),
          nn.MaxPool2d(kernel_size=2,
                        stride=2) # default stride value is same as kernel_size
      )
      self.conv_block_2 = nn.Sequential(
          nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
          nn.ReLU(),
          nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
          nn.ReLU(),
          nn.MaxPool2d(2)
      )
      self.classifier = nn.Sequential(
          nn.Flatten(),
          # Where did this in_features shape come from?
          # It's because each layer of our network compresses and changes the shape of our inputs data.
          nn.Linear(in_features=hidden_units*13*13,
                    out_features=output_shape)
      )

      self.layer_stack=nn.Sequential(
          self.conv_block_1,
          self.conv_block_2,
          self.classifier
      )

  def forward(self, x: torch.Tensor):
      return self.layer_stack(x)
      # return self.classifier(self.block_2(self.block_1(x))) # <- leverage the benefits of operator fusion

Writing going_modular/model_builder.py


In [12]:
import torch

In [13]:
from going_modular import model_builder

device="cuda" if torch.cuda.is_available() else "cpu"

#setup model_1
torch.manual_seed(42)
model_1 = model_builder.TinyVGG(input_shape=3,
                              hidden_units=10,
                             output_shape=len(class_names)).to(device)

In [14]:
model_1

TinyVGG(
  (conv_block_1): Sequential(
    (0): Conv2d(3, 10, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block_2): Sequential(
    (0): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1690, out_features=3, bias=True)
  )
  (layer_stack): Sequential(
    (0): Sequential(
      (0): Conv2d(3, 10, kernel_size=(3, 3), stride=(1, 1))
      (1): ReLU()
      (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1))
      (3): ReLU()
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
 

In [15]:
# 1. Get a batch of images and labels from the DataLoader
img_batch, label_batch = next(iter(train_dataloader))

img_batch, label_batch

(tensor([[[[0.3922, 0.3922, 0.4000,  ..., 0.3843, 0.3843, 0.3725],
           [0.4078, 0.4000, 0.4078,  ..., 0.3882, 0.4000, 0.3961],
           [0.4039, 0.4078, 0.4078,  ..., 0.4039, 0.4118, 0.4078],
           ...,
           [0.5451, 0.3922, 0.5961,  ..., 0.5255, 0.4706, 0.4118],
           [0.5412, 0.3569, 0.5333,  ..., 0.5294, 0.5333, 0.5294],
           [0.5843, 0.4431, 0.2784,  ..., 0.5216, 0.5294, 0.5294]],
 
          [[0.2431, 0.2275, 0.2275,  ..., 0.2353, 0.2431, 0.2392],
           [0.2431, 0.2314, 0.2353,  ..., 0.2549, 0.2627, 0.2627],
           [0.2275, 0.2353, 0.2471,  ..., 0.2745, 0.2667, 0.2627],
           ...,
           [0.5176, 0.3529, 0.5647,  ..., 0.4863, 0.4275, 0.3725],
           [0.5098, 0.3176, 0.4980,  ..., 0.4902, 0.4902, 0.4902],
           [0.5490, 0.4118, 0.2353,  ..., 0.4824, 0.4863, 0.4784]],
 
          [[0.1137, 0.1294, 0.1294,  ..., 0.1804, 0.1882, 0.1725],
           [0.1255, 0.1333, 0.1176,  ..., 0.1922, 0.1922, 0.2039],
           [0.1098, 0.13

In [16]:
# 2. Get a single image from the batch and unsqueeze the image so its shape fits the model
img_single, label_single = img_batch[0].unsqueeze(dim=0), label_batch[0]
print(f"Single image shape: {img_single.shape}\n")
print(f"Ground Truth: {label_single}\n")

Single image shape: torch.Size([1, 3, 64, 64])

Ground Truth: 0



In [17]:
# 3. Perform a forward pass on a single image
model_1.eval()
with torch.inference_mode():
    pred = model_1(img_single.to(device))


# 4. Print out what's happening and convert model logits -> pred probs -> pred label
print(f"Output logits:\n{pred}\n")
print(f"Output prediction probabilities:\n{torch.softmax(pred, dim=1)}\n")
print(f"Output prediction label:\n{torch.argmax(torch.softmax(pred, dim=1), dim=1)}\n")
print(f"Actual label:\n{label_single}")

Output logits:
tensor([[ 0.0208, -0.0020,  0.0095]], device='cuda:0')

Output prediction probabilities:
tensor([[0.3371, 0.3295, 0.3333]], device='cuda:0')

Output prediction label:
tensor([0], device='cuda:0')

Actual label:
0


In [18]:
%%writefile going_modular/engine.py
"""
contains functions for training and testing a PyTorch model
"""

from typing import Dict,List,Tuple

import torch

from tqdm.auto import tqdm

def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device: torch.device) -> Tuple[float, float]:
    """Trains a PyTorch model for a single epoch.

    Turns a target PyTorch model to training mode and then
    runs through all of the required training steps (forward
    pass, loss calculation, optimizer step).

    Args:
    model: A PyTorch model to be trained.
    dataloader: A DataLoader instance for the model to be trained on.
    loss_fn: A PyTorch loss function to minimize.
    optimizer: A PyTorch optimizer to help minimize the loss function.
    device: A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
    A tuple of training loss and training accuracy metrics.
    In the form (train_loss, train_accuracy). For example:

    (0.1112, 0.8743)
    """
    # Put model in train mode
    model.train()

    # Setup train loss and train accuracy values
    train_loss, train_acc = 0, 0

    # Loop through data loader data batches
    for batch, (X, y) in enumerate(dataloader):
        # Send data to target device
        X, y = X.to(device), y.to(device)

        # 1. Forward pass
        y_pred = model(X)

        # 2. Calculate  and accumulate loss
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        # 3. Optimizer zero grad
        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer step
        optimizer.step()

        # Calculate and accumulate accuracy metric across all batches
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item()/len(y_pred)

    # Adjust metrics to get average loss and accuracy per batch
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc

def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device: torch.device) -> Tuple[float, float]:
    """Tests a PyTorch model for a single epoch.

    Turns a target PyTorch model to "eval" mode and then performs
    a forward pass on a testing dataset.

    Args:
    model: A PyTorch model to be tested.
    dataloader: A DataLoader instance for the model to be tested on.
    loss_fn: A PyTorch loss function to calculate loss on the test data.
    device: A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
    A tuple of testing loss and testing accuracy metrics.
    In the form (test_loss, test_accuracy). For example:

    (0.0223, 0.8985)
    """
    # Put model in eval mode
    model.eval()

    # Setup test loss and test accuracy values
    test_loss, test_acc = 0, 0

    # Turn on inference context manager
    with torch.inference_mode():
        # Loop through DataLoader batches
        for batch, (X, y) in enumerate(dataloader):
            # Send data to target device
            X, y = X.to(device), y.to(device)

            # 1. Forward pass
            test_pred_logits = model(X)

            # 2. Calculate and accumulate loss
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()

            # Calculate and accumulate accuracy
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += ((test_pred_labels == y).sum().item()/len(test_pred_labels))

    # Adjust metrics to get average loss and accuracy per batch
    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss,test_acc

def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device) -> Dict[str, List[float]]:
    """Trains and tests a PyTorch model.

    Passes a target PyTorch models through train_step() and test_step()
    functions for a number of epochs, training and testing the model
    in the same epoch loop.

    Calculates, prints and stores evaluation metrics throughout.

    Args:
    model: A PyTorch model to be trained and tested.
    train_dataloader: A DataLoader instance for the model to be trained on.
    test_dataloader: A DataLoader instance for the model to be tested on.
    optimizer: A PyTorch optimizer to help minimize the loss function.
    loss_fn: A PyTorch loss function to calculate loss on both datasets.
    epochs: An integer indicating how many epochs to train for.
    device: A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
    A dictionary of training and testing loss as well as training and
    testing accuracy metrics. Each metric has a value in a list for
    each epoch.
    In the form: {train_loss: [...],
              train_acc: [...],
              test_loss: [...],
              test_acc: [...]}
    For example if training for epochs=2:
             {train_loss: [2.0616, 1.0537],
              train_acc: [0.3945, 0.3945],
              test_loss: [1.2641, 1.5706],
              test_acc: [0.3400, 0.2973]}
    """
    # Create empty results dictionary
    results = {"train_loss": [],
               "train_acc": [],
               "test_loss": [],
               "test_acc": []
    }

    # Loop through training and testing steps for a number of epochs
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                          dataloader=train_dataloader,
                                          loss_fn=loss_fn,
                                          optimizer=optimizer,
                                          device=device)
        test_loss, test_acc = test_step(model=model,
          dataloader=test_dataloader,
          loss_fn=loss_fn,
          device=device)

        # Print out what's happening
        print(
          f"Epoch: {epoch+1} | "
          f"train_loss: {train_loss:.4f} | "
          f"train_acc: {train_acc:.4f} | "
          f"test_loss: {test_loss:.4f} | "
          f"test_acc: {test_acc:.4f}",
          flush= True
        )

        # Update results dictionary
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

    # Return the filled results at the end of the epochs
    return results

Writing going_modular/engine.py


In [19]:
%%writefile going_modular/utils.py
from pathlib import Path
import torch

def save_model(model: torch.nn.Module,
               target_dir: str,
               model_name: str):
  """Saves a PyTorch model to a target directory.

  Args:
    model: A target PyTorch model to save.
    target_dir: A directory for saving the model to.
    model_name: A filename for the saved model. Should include
      either ".pth" or ".pt" as the file extension.

  Example usage:
    save_model(model=model_0,
               target_dir="models",
               model_name="05_going_modular_tingvgg_model.pth")
  """
  # Create target directory
  target_dir_path = Path(target_dir)
  target_dir_path.mkdir(parents=True,
                        exist_ok=True)

  # Create model save path
  assert model_name.endswith(".pth") or model_name.endswith(".pt"), "model_name should end with '.pt' or '.pth'"
  model_save_path = target_dir_path / model_name

  # Save the model state_dict()
  print(f"[INFO] Saving model to: {model_save_path}")
  torch.save(obj=model.state_dict(),
             f=model_save_path)


Writing going_modular/utils.py


In [20]:
%%writefile going_modular/train.py
"""
Trains a PyTorch image classification model using device-agnostic code.
"""
import os
import argparse
import torch
from torchvision import transforms
import data_setup, engine, model_builder, utils

def get_args():
    parser = argparse.ArgumentParser(description="Train TinyVGG model")

    parser.add_argument("--epochs",       type=int,   default=5,     help="Number of epochs")
    parser.add_argument("--batch_size",   type=int,   default=32,    help="Batch size")
    parser.add_argument("--hidden_units", type=int,   default=10,    help="Hidden units in TinyVGG")
    parser.add_argument("--lr",           type=float, default=0.001, help="Learning rate")
    parser.add_argument("--train_dir",    type=str,   default="data/pizza_steak_sushi/train")
    parser.add_argument("--test_dir",     type=str,   default="data/pizza_steak_sushi/test")

    return parser.parse_args()

def main():
    args = get_args()

    # Print what we're using
    print(f"[INFO] Epochs:       {args.epochs}")
    print(f"[INFO] Batch size:   {args.batch_size}")
    print(f"[INFO] Hidden units: {args.hidden_units}")
    print(f"[INFO] LR:           {args.lr}")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[INFO] Using device: {device}")

    data_transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor()
    ])

    train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
        train_dir=args.train_dir,
        test_dir=args.test_dir,
        transform=data_transform,
        batch_size=args.batch_size       # 👈 from args
    )

    model = model_builder.TinyVGG(
        input_shape=3,
        hidden_units=args.hidden_units,  # 👈 from args
        output_shape=len(class_names)
    ).to(device)

    loss_fn   = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)  # 👈 from args

    engine.train(
        model=model,
        train_dataloader=train_dataloader,
        test_dataloader=test_dataloader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        epochs=args.epochs,              # 👈 from args
        device=device
    )

    utils.save_model(
        model=model,
        target_dir="models",
        model_name="05_going_modular_script_mode_tinyvgg_model.pth"
    )

if __name__ == "__main__":
    main()

Writing going_modular/train.py


In [21]:
!python -u going_modular/train.py --epochs 10 --batch_size 64 --hidden_units 20 --lr 0.0001


[INFO] Epochs:       10
[INFO] Batch size:   64
[INFO] Hidden units: 20
[INFO] LR:           0.0001
[INFO] Using device: cuda
  0% 0/10 [00:00<?, ?it/s]Epoch: 1 | train_loss: 1.0991 | train_acc: 0.3411 | test_loss: 1.1056 | test_acc: 0.1953
 10% 1/10 [00:01<00:17,  1.99s/it]Epoch: 2 | train_loss: 1.0975 | train_acc: 0.3600 | test_loss: 1.1035 | test_acc: 0.1953
 20% 2/10 [00:03<00:13,  1.73s/it]Epoch: 3 | train_loss: 1.0974 | train_acc: 0.3414 | test_loss: 1.1077 | test_acc: 0.1953
 30% 3/10 [00:04<00:10,  1.57s/it]Epoch: 4 | train_loss: 1.0966 | train_acc: 0.3377 | test_loss: 1.1094 | test_acc: 0.1953
 40% 4/10 [00:05<00:07,  1.31s/it]Epoch: 5 | train_loss: 1.0956 | train_acc: 0.3377 | test_loss: 1.1106 | test_acc: 0.1953
 50% 5/10 [00:06<00:05,  1.18s/it]Epoch: 6 | train_loss: 1.0942 | train_acc: 0.3639 | test_loss: 1.1097 | test_acc: 0.2109
 60% 6/10 [00:08<00:04,  1.22s/it]Epoch: 7 | train_loss: 1.0934 | train_acc: 0.3648 | test_loss: 1.1086 | test_acc: 0.2188
 70% 7/10 [00:09<00:0